# 52 Pairwise inter-landmark distances

Deletion preserves every non-masked vertex bit-exact, so the ten pairwise distances among the five 10--20 landmarks must match between the original and anonymized meshes. This notebook computes that table for the representative subject (S2 = Subject 17) and populates Table 4.3 of the thesis.

**Cohort scope.** Reported for one representative subject because the deletion operator is the same for every subject in the eleven-subject cohort (S1--S7 optode-cap, S8--S11 bare-cap; Subject 11 is excluded as an unusable acquisition); the remaining ten entries in Table 4.3 would be identical zeros and add no information. The cohort-wide claim that nearest-vertex identity is preserved on every subject is established by notebook 51's bidirectional sweep (§4.4.2) and notebook 53's KDTree check (§4.4.1).

Output: `thesis_results_out/pairwise_distances.csv`.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
from _thesis_helpers import (
    SUBJECTS, subject_paths, load_raw, load_anon, load_landmarks,
    run_pipeline, is_optode, s_id,
)

import numpy as np
import pandas as pd

OUT_DIR = pathlib.Path('thesis_results_out')
OUT_DIR.mkdir(exist_ok=True)

# Representative subject for the single-subject pairwise table.
SUBJECT = 17
print(f'Pairwise distances for {s_id(SUBJECT)} (Subject{SUBJECT}, '
      f'optode={is_optode(SUBJECT)})')

Pairwise distances for S2 (Subject17, optode=True)


## 1. Load original and anonymized landmarks

The TSV written by `save_anonymized_scan` uses the digitized frame, the same frame as the raw OBJ. Both the original and anonymized landmarks therefore live in a common coordinate system; any difference between pairwise distances reflects landmark displacement, not a frame mismatch.

In [2]:
import cedalion.io

paths = subject_paths(SUBJECT)

landmarks_orig = cedalion.io.load_tsv(str(paths.landmarks_tsv))

print('Original landmarks:')

print(landmarks_orig)

Original landmarks:
<xarray.DataArray (label: 5, digitized: 3)> Size: 120B
<Quantity([[182.89526  117.989616 432.334997]
 [ 53.406784 217.899417 468.502655]
 [ 72.097205  88.432967 573.862275]
 [ 71.591662 -13.006238 478.993564]
 [  5.307631  46.375214 419.38877 ]], 'millimeter')>
Coordinates:
  * label    (label) object 40B 'Cz' 'Iz' 'RPA' 'Nz' 'LPA'
    type     (label) object 40B PointType.LANDMARK ... PointType.LANDMARK
Dimensions without coordinates: digitized


## 2. Re-run pipeline to obtain the anonymized landmarks

`save_anonymized_scan` writes a single `_landmarks.tsv` -- the landmarks as they sit on the anonymized output. For this comparison we want both: the landmarks on the original (just loaded) and the landmarks after the affine round-trip.  Since the deletion step itself does not touch the landmark array, we can simply re-run the pipeline and take `landmarks_dig`.

In [3]:
surface_raw = load_raw(SUBJECT)

art = run_pipeline(surface_raw, landmarks_orig, subject=SUBJECT)

landmarks_anon = art.landmarks_dig

print('Anonymized landmarks:')

print(landmarks_anon)

Anonymized landmarks:
<xarray.DataArray (label: 5, digitized: 3)> Size: 120B
<Quantity([[182.89526  117.989616 432.334997]
 [ 53.406784 217.899417 468.502655]
 [ 72.097205  88.432967 573.862275]
 [ 71.591662 -13.006238 478.993564]
 [  5.307631  46.375214 419.38877 ]], 'millimeter')>
Coordinates:
    type     (label) object 40B PointType.LANDMARK ... PointType.LANDMARK
  * label    (label) <U3 60B 'Cz' 'Iz' 'RPA' 'Nz' 'LPA'
Dimensions without coordinates: digitized


## 3. Compute pairwise distances

All 10 unordered pairs among {Nz, Iz, Cz, Lpa, Rpa}.

In [4]:
from itertools import combinations



def _pos(da, label):

    arr = da.pint.dequantify().values

    labels = list(da['label'].values)

    return arr[labels.index(label)]



labels = ['Nz', 'Iz', 'Cz', 'LPA', 'RPA']

rows = []

for a, b in combinations(labels, 2):

    d_orig = float(np.linalg.norm(_pos(landmarks_orig, a) - _pos(landmarks_orig, b)))

    d_anon = float(np.linalg.norm(_pos(landmarks_anon, a) - _pos(landmarks_anon, b)))

    rows.append({

        'pair': f'{a}-{b}',

        'd_original_mm': d_orig,

        'd_anonymized_mm': d_anon,

        'abs_diff_mm': abs(d_orig - d_anon),

    })



df = pd.DataFrame(rows)

df

,pair,d_original_mm,d_anonymized_mm,abs_diff_mm
0,Nz-Iz,231.858083,231.858083,0.000000e+00
1,Nz-Cz,178.116329,178.116329,0.000000e+00
2,Nz-LPA,107.109575,107.109575,0.000000e+00
3,Nz-RPA,138.889309,138.889309,2.842171e-14
4,Iz-Cz,167.503234,167.503234,2.842171e-14
5,Iz-LPA,184.787052,184.787052,0.000000e+00
6,Iz-RPA,167.962922,167.962922,5.684342e-14
7,Cz-LPA,191.920800,191.920800,0.000000e+00
8,Cz-RPA,182.153163,182.153163,5.684342e-14
9,LPA-RPA,173.469783,173.469783,5.684342e-14


## 4. Save

In [5]:
out = OUT_DIR / 'pairwise_distances.csv'

df.to_csv(out, index=False)

print(f'Max |diff|: {df.abs_diff_mm.max():.6g} mm')

print(f'Wrote {out}')

Max |diff|: 5.68434e-14 mm
Wrote thesis_results_out/pairwise_distances.csv
